运行约定：以本 Notebook 所在目录作为工作目录，从第一单元顺序运行。首个代码单元仅建立数据与输出目录。历史输出保留，不代表当前环境已验证。


In [ ]:
from pathlib import Path
Path('../../data').mkdir(parents=True, exist_ok=True)
Path('../../outputs/pytorch').mkdir(parents=True, exist_ok=True)


In [ ]:
# encoding=utf8
import os
import sys
import math
import random
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F

# 设置随机种子，保证结果可复现
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# 词向量：迈向NLP领域的第1步台阶

> **本 Notebook 为 PyTorch 版本**，核心计算均使用 PyTorch 张量操作完成。

## 1. 词向量概述

在自然语言处理任务中，词向量（Word Embedding）是表示自然语言里单词的一种方法，即把每个词都表示为一个N维空间内的点，即一个高维空间内的向量。通过这种方法，实现把自然语言计算转换为向量计算。

如 **图1** 所示的词向量计算任务中，先把每个词（如queen，king等）转换成一个高维空间的向量，这些向量在一定意义上可以代表这个词的语义信息。再通过计算这些向量之间的距离，就可以计算出词语之间的关联关系，从而达到让计算机像计算数值一样去计算自然语言的目的。

<center><img src="https://ai-studio-static-online.cdn.bcebos.com/00ba55f7304e4f97942165cf1deb946ced404a19325d40969b7c220e30cf527e" width="800" ></center>
<center>图1：词向量计算示意图</center>
<br></br>

因此，大部分词向量模型都需要回答两个问题：

1. **如何把词转换为向量?**

自然语言单词是离散信号，比如"香蕉"，"橘子"，"水果"在我们看来就是3个离散的词。

如何把每个离散的单词转换为一个向量？

2. **如何让向量具有语义信息?**

比如，我们知道在很多情况下，"香蕉"和"橘子"更加相似，而"香蕉"和"句子"就没有那么相似，同时"香蕉"和"食物"、"水果"的相似程度可能介于"橘子"和"句子"之间。

那么，我们该如何让词向量具备这样的语义信息？

### 1.1 如何把词转换为向量

自然语言单词是离散信号，比如"我"、" 爱"、"人工智能"。如何把每个离散的单词转换为一个向量？通常情况下，我们可以维护一个如 **图2** 所示的查询表。表中每一行都存储了一个特定词语的向量值，每一列的第一个元素都代表着这个词本身，以便于我们进行词和向量的映射（如"我"对应的向量值为 [0.3，0.5，0.7，0.9，-0.2，0.03] ）。给定任何一个或者一组单词，我们都可以通过查询这个excel，实现把单词转换为向量的目的，这个查询和替换过程称之为Embedding Lookup。
<center><img src="https://ai-studio-static-online.cdn.bcebos.com/2ccf57a0c4584ca5b5000da85bc48c4cbe1588f0e3f548c6b2271c26e2e7df14" width="800" ></center>
<center><br>图2：词向量查询表</br></center>
<br></br>

上述过程也可以使用一个字典数据结构实现。事实上如果不考虑计算效率，使用字典实现上述功能是个不错的选择。然而在进行神经网络计算的过程中，需要大量的算力，常常要借助特定硬件（如GPU）满足训练速度的需求。GPU上所支持的计算都是以张量（Tensor）为单位展开的，因此在实际场景中，我们需要把Embedding Lookup的过程转换为张量计算，如 **图3** 所示。
<center><img src="https://ai-studio-static-online.cdn.bcebos.com/2145ed77e5b24e0b9b33a7c8451b52cdc68f38b651494b51afdd264724392fe3" width="800" ></center>
<center><br>图3：张量计算示意图</br></center>
<br></br>

假设对于句子"我，爱，人工，智能"，把Embedding Lookup的过程转换为张量计算的流程如下：

1. 通过查询字典，先把句子中的单词转换成一个ID（通常是一个大于等于0的整数），这个单词到ID的映射关系可以根据需求自定义（如**图3**中，我=>1, 人工=>2，爱=>3，...）。

2. 得到ID后，再把每个ID转换成一个固定长度的向量。假设字典的词表中有5000个词，那么，对于单词"我"，就可以用一个5000维的向量来表示。由于"我"的ID是1，因此这个向量的第一个元素是1，其他元素都是0（[1，0，0，…，0]）；同样对于单词"人工"，第二个元素是1，其他元素都是0。用这种方式就实现了用一个向量表示一个单词。由于每个单词的向量表示都只有一个元素为1，而其他元素为0，因此我们称上述过程为One-Hot Encoding。

3. 经过One-Hot Encoding后，句子"我，爱，人工，智能"就被转换成为了一个形状为 4×5000的张量，记为$V$。在这个张量里共有4行、5000列，从上到下，每一行分别代表了"我"、"爱"、"人工"、"智能"四个单词的One-Hot Encoding。最后，我们把这个张量$V$和另外一个稠密张量$W$相乘，其中$W$张量的形状为5000 × 128（5000表示词表大小，128表示每个词的向量大小）。经过张量乘法，我们就得到了一个4×128的张量，从而完成了把单词表示成向量的目的。

### 1.2 如何让向量具有语义信息

得到每个单词的向量表示后，我们需要思考下一个问题：比如在多数情况下，"香蕉"和"橘子"更加相似，而"香蕉"和"句子"就没有那么相似；同时，"香蕉"和"食物"、"水果"的相似程度可能介于"橘子"和"句子"之间。那么如何让存储的词向量具备这样的语义信息呢？

我们先学习自然语言处理领域的一个小技巧。在自然语言处理研究中，科研人员通常有一个共识：使用一个单词的上下文来了解这个单词的语义，比如：

 >"苹果手机质量不错，就是价格有点贵。"
 >
 >"这个苹果很好吃，非常脆。"
 >
 >"菠萝质量也还行，但是不如苹果支持的APP多。"
>
在上面的句子中，我们通过上下文可以推断出第一个"苹果"指的是苹果手机，第二个"苹果"指的是水果苹果，而第三个"菠萝"指的应该也是一个手机。事实上，在自然语言处理领域，使用上下文描述一个词语或者元素的语义是一个常见且有效的做法。我们可以使用同样的方式训练词向量，让这些词向量具备表示语义信息的能力。

## 2. word2vec 词向量训练算法

2013年，Mikolov提出的经典word2vec算法就是通过上下文来学习语义信息。word2vec包含两个经典模型：CBOW（Continuous Bag-of-Words）和Skip-gram，如 **图4** 所示。

- **CBOW**：通过上下文的词向量推理中心词。
- **Skip-gram**：根据中心词推理上下文。

<center><img src="https://ai-studio-static-online.cdn.bcebos.com/87b90136eef04d7285803e567a5f7f6d0e40bb552deb40a19a7540c4e6aa20a3" width="700" ></center>
<center><br>图4：CBOW和Skip-gram语义学习示意图</br></center>
<br></br>

假设有一个句子"Pineapples are spiked and yellow"，两个模型的推理方式如下：

- 在**CBOW**中，先在句子中选定一个中心词，并把其它词作为这个中心词的上下文。如 **图4** CBOW所示，把"Spiked"作为中心词，把"Pineapples、are、and、yellow"作为中心词的上下文。在学习过程中，使用上下文的词向量推理中心词，这样中心词的语义就被传递到上下文的词向量中，如"Spiked → pineapple"，从而达到学习语义信息的目的。

- 在**Skip-gram**中，同样先选定一个中心词，并把其他词作为这个中心词的上下文。如 **图4** Skip-gram所示，把"Spiked"作为中心词，把"Pineapples、are、and、yellow"作为中心词的上下文。不同的是，在学习过程中，使用中心词的词向量去推理上下文，这样上下文定义的语义被传入中心词的表示中，如"pineapple → Spiked"，
从而达到学习语义信息的目的。

------
**说明：**

一般来说，CBOW比Skip-gram训练速度快，训练过程更加稳定，原因是CBOW使用上下文average的方式进行训练，每个训练step会见到更多样本。而在生僻字（出现频率低的字）处理上，skip-gram比CBOW效果更好，原因是skip-gram不会刻意回避生僻字(CBOW结构中输入中存在生僻字时，生僻字会被其它非生僻字的权重冲淡)。

------

## 3. 使用 PyTorch 加载与操作词向量

在 PyTorch 中，我们可以使用 `torch.nn.Embedding` 来构建词向量层。对于预训练好的词向量（如 word2vec、glove 等），可以将其权重加载到 `Embedding` 层中，从而复用已经训练好的语义信息。

本节将演示：
1. 使用 `gensim` 训练/加载中文词向量（框架无关，仅用于获取向量权重）
2. 将其转换为 `torch.nn.Embedding`
3. 使用 PyTorch 张量操作进行词向量检索、相似度计算和可视化

### 3.1 准备数据并训练（或加载）词向量

为了便于演示，我们先下载 ChnSentiCorp 数据集，并使用 `gensim` 在其上快速训练一个轻量级中文词向量。在实际生产中，建议使用更大规模语料训练的词向量（如腾讯词向量、百度百科词向量等）。

In [ ]:
import os
import urllib.request
import zipfile

# 下载 ChnSentiCorp 数据集（用于训练演示用的词向量）
data_url = "https://bj.bcebos.com/paddlenlp/datasets/ChnSentiCorp.zip"
data_dir = "../../data"
zip_path = os.path.join(data_dir, "ChnSentiCorp.zip")

os.makedirs(data_dir, exist_ok=True)
if not os.path.exists(zip_path):
    print("Downloading ChnSentiCorp dataset...")
    urllib.request.urlretrieve(data_url, zip_path)
    print("Download done.")
else:
    print("Dataset already exists.")

# 解压
extract_dir = os.path.join(data_dir, "ChnSentiCorp")
if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(data_dir)
    print("Extract done.")
else:
    print("Dataset already extracted.")

In [ ]:
import jieba
from collections import defaultdict

# 读取训练集文本，用于训练 word2vec
train_path = os.path.join(data_dir, "ChnSentiCorp", "train.tsv")
sentences = []
with open(train_path, 'r', encoding='utf-8') as f:
    head = f.readline()  # skip header
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 2:
            text = parts[-1]
            words = list(jieba.cut(text))
            sentences.append(words)

print(f"Total sentences for training: {len(sentences)}")
print(f"Example: {sentences[0][:20]}")

In [ ]:
from gensim.models import Word2Vec

# 训练一个轻量级中文词向量（实际项目中请使用大规模预训练词向量）
print("Training Word2Vec model...")
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=128,   # 向量维度
    window=5,          # 上下文窗口
    min_count=5,       # 最小词频
    workers=4,
    sg=1,              # 1=Skip-gram, 0=CBOW
    seed=SEED
)
print("Training done.")
print(f"Vocab size: {len(w2v_model.wv)}")

# 测试相似词
if '酒店' in w2v_model.wv:
    print("Similar words to '酒店':")
    for w, s in w2v_model.wv.most_similar('酒店', topn=5):
        print(f"  {w}: {s:.4f}")

### 3.2 将 gensim 词向量加载到 PyTorch Embedding

`gensim` 训练好的词向量是 NumPy 数组格式，我们可以按如下步骤转换为 PyTorch 的 `nn.Embedding`：
1. 构建词表（word2idx）
2. 提取权重矩阵（vocab_size × embedding_dim）
3. 创建 `nn.Embedding` 并加载权重

In [ ]:
# 构建词表
word2idx = {"[PAD]": 0, "[UNK]": 1}
embed_dim = w2v_model.wv.vector_size

# 从 gensim 模型中提取词和向量
vectors = []
# PAD vector = zeros
vectors.append(np.zeros(embed_dim, dtype=np.float32))
# UNK vector = random normal
vectors.append(np.random.randn(embed_dim).astype(np.float32) * 0.01)

for word in w2v_model.wv.index_to_key:
    word2idx[word] = len(word2idx)
    vectors.append(w2v_model.wv[word].astype(np.float32))

idx2word = {v: k for k, v in word2idx.items()}
vocab_size = len(word2idx)
weights_matrix = np.stack(vectors)  # shape: [vocab_size, embed_dim]

print(f"Vocab size: {vocab_size}, Embedding dim: {embed_dim}")
print(f"Weights matrix shape: {weights_matrix.shape}")

# 创建 PyTorch Embedding 层并加载预训练权重
embedding_layer = nn.Embedding.from_pretrained(
    torch.FloatTensor(weights_matrix),
    freeze=False,   # 允许微调
    padding_idx=0   # [PAD] 对应索引 0
)
print("PyTorch Embedding layer created.")

### 3.3 检索词向量

基于构建好的 `word2idx` 和 `embedding_layer`，我们可以轻松检索任意单词的 PyTorch 词向量。

In [ ]:
def get_word_embedding(word):
    """获取单个词的 PyTorch 词向量"""
    idx = word2idx.get(word, word2idx["[UNK]"])
    idx_tensor = torch.LongTensor([idx])
    return embedding_layer(idx_tensor)  # shape: [1, embed_dim]

# 测试：检索"中国"的词向量（如果在词表中）
test_word = "酒店" if "酒店" in word2idx else list(word2idx.keys())[10]
emb = get_word_embedding(test_word)
print(f"Word: {test_word}")
print(f"Embedding shape: {emb.shape}")
print(f"Embedding (first 10 dims): {emb[0, :10]}")

### 3.4 计算词向量相似度

PyTorch 提供了丰富的张量运算 API，可以方便地计算内积（dot）和余弦相似度（cosine similarity）。

In [ ]:
def word_dot(word1, word2):
    """计算两个词向量的内积"""
    emb1 = get_word_embedding(word1)
    emb2 = get_word_embedding(word2)
    return torch.matmul(emb1, emb2.T).item()

def word_cosine_sim(word1, word2):
    """计算两个词向量的余弦相似度"""
    emb1 = get_word_embedding(word1)
    emb2 = get_word_embedding(word2)
    return F.cosine_similarity(emb1, emb2).item()

# 示例（请确保测试词在词表中）
for w1, w2 in [("房间", "酒店"), ("房间", "书本")]:
    if w1 in word2idx and w2 in word2idx:
        dot_score = word_dot(w1, w2)
        cos_score = word_cosine_sim(w1, w2)
        print(f"[{w1}] vs [{w2}] | dot={dot_score:.4f}, cosine={cos_score:.4f}")

### 3.5 词向量可视化

PyTorch 生态中通常使用 **TensorBoard** 的 `add_embedding` 进行高维向量可视化。执行以下代码后，可在命令行启动 TensorBoard 查看：

```bash
tensorboard --logdir ../../outputs/pytorch/runs
```

In [ ]:
from torch.utils.tensorboard import SummaryWriter

# 选取前 1000 个词进行可视化
num_vis = min(1000, vocab_size)
vis_words = [idx2word[i] for i in range(num_vis)]
vis_embeddings = embedding_layer(torch.arange(num_vis)).detach().numpy()

writer = SummaryWriter(log_dir="../../outputs/pytorch/runs/word2vec_vis")
writer.add_embedding(vis_embeddings, metadata=vis_words, tag="word2vec")
writer.close()
print("VisualDL / TensorBoard log saved to ../../outputs/pytorch/runs/word2vec_vis")
print("Run: tensorboard --logdir ../../outputs/pytorch/runs")

### 3.6 基于词向量计算句子间的相似度

前边讲了如何使用词向量进行词向量级的计算，那么我们应该如何使用这些词向量计算两个句子之间的相似度呢。

计算句子间的相似度的一般做法是先获取句子的句向量表示，然后根据句向量进行相似度计算，如cosine相似度。

#### 句向量表示

对于给定的一句话，一种简单的做法是首先对给定的语句进行分词，然后将每个词的词向量进行相加并平均，作为这句话的句向量。

例如， 给定语句："飞桨是优秀的深度学习平台"，可以通过执行如下操作获取句向量：
- 语句分词：['飞桨', '是', '优秀', '的', '深度', '学习', '平台']
- 获取每个单词的词向量$V = [v_1, v_2, v_3, v_4, v_5, v_6, v_7]$
- 将词向量进行相加 $v = \sum_1^7 v_i$
- 将获取的向量$v$进行平均得到句向量： $v = v/7$

In [ ]:
def get_sentence_embedding(text):
    """基于平均词向量计算句向量"""
    words = list(jieba.cut(text))
    # 过滤掉未登录词（可选）
    indices = [word2idx.get(w, word2idx["[UNK]"]) for w in words]
    if len(indices) == 0:
        return torch.zeros(embed_dim)
    idx_tensor = torch.LongTensor(indices)
    word_embs = embedding_layer(idx_tensor)  # [seq_len, embed_dim]
    sent_emb = word_embs.mean(dim=0)         # [embed_dim]
    return sent_emb

# 示例
for text in ["房间很干净", "卫生条件不错"]:
    emb = get_sentence_embedding(text)
    print(f"Text: {text} | Embedding shape: {emb.shape}")

In [ ]:
# 计算两句话的 cosine 相似度
def sentence_cosine_sim(text1, text2):
    emb1 = get_sentence_embedding(text1).unsqueeze(0)
    emb2 = get_sentence_embedding(text2).unsqueeze(0)
    return F.cosine_similarity(emb1, emb2).item()

text_pairs = [
    ("这家酒店服务很好", "这家酒店服务很好"),
    ("这家酒店服务很好", "这家餐厅菜很难吃"),
]

for t1, t2 in text_pairs:
    sim = sentence_cosine_sim(t1, t2)
    print(f"Similarity: {sim:.5f}\n  '{t1}'\n  '{t2}'\n")

## 4. 常见 word2vec 预训练模型

### 4.1 英文预训练模型

| 模型                       | 语料                                      | 维度                   | 词汇量        | 下载地址                                                                                 |
| ------------------------ | --------------------------------------- | -------------------- | ---------- | ------------------------------------------------------------------------------------ |
| **Google News Word2Vec** | Google News (约 100B 词)                  | 300                  | ~300万词/短语 | [GoogleNews-vectors-negative300.bin.gz](https://code.google.com/archive/p/word2vec/) |
| **GloVe**                | Wikipedia + Gigaword / Common Crawl     | 50 / 100 / 200 / 300 | 数十亿词       | [GloVe 官方](https://nlp.stanford.edu/projects/glove/)                                 |
| **FastText (English)**   | Wikipedia / Common Crawl / UMBC webbase | 300                  | 数百万词       | [FastText 官方](https://fasttext.cc/docs/en/english-vectors.html)                      |

### 4.2 中文预训练模型

| 模型                         | 语料                | 维度        | 词汇量   | 下载地址                                                                 |
| -------------------------- | ----------------- | --------- | ----- | -------------------------------------------------------------------- |
| **腾讯词向量 (Tencent AI Lab)** | 海量中文互联网语料         | 200       | 800万  | [Tencent Embedding](https://ai.tencent.com/ailab/nlp/embedding.html) |
| **北大中文 Word2Vec**          | 中文 Wikipedia + 新闻 | 300       | 数十万   | GitHub 第三方整理版                                                        |
| **中文 Wikipedia Word2Vec**  | 中文 Wikipedia      | 100 / 300 | ~几十万 | [部分 GitHub 资源](https://github.com/Embedding/Chinese-Word-Vectors)    |
| **FastText 中文**            | Wikipedia / 新闻语料  | 300       | 数百万   | [FastText 中文](https://fasttext.cc/docs/en/crawl-vectors.html)        |

## 5. 小结

在本节中，我们学习了：
- 词向量的基本概念和作用
- word2vec 的两种训练模式：CBOW 和 Skip-gram
- 如何使用 `gensim` 获取/训练词向量，并将其加载到 PyTorch 的 `nn.Embedding` 中
- 使用 PyTorch 张量运算计算词/句级别的 cosine 相似度
- 使用 TensorBoard 进行词向量可视化

下一节将基于这些基础知识，使用 **LSTM** 完成完整的中文情感分类任务。